In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q nlpaug transformers torch sacremoses


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 37.0 MB/s eta 0:00:00


In [2]:
import pandas as pd
import torch
import random
from transformers import pipeline
from tqdm import tqdm

In [3]:
# 1. Load your existing combined dataset
df = pd.read_csv("/content/drive/MyDrive/AIE_Project/intent_classification_datasets/combined_dataset.csv")
print(f"Original dataset size: {len(df)} rows.")

# 2. Set device (GPU if available, otherwise CPU)
device = 0 if torch.cuda.is_available() else -1

# 3. Initialize a standard Hugging Face Masked Language Model pipeline
print("Loading text augmentation model...")
mask_filler = pipeline("fill-mask", model="distilbert-base-uncased", device=device)

augmented_records = []

print("Generating synthetic variants... (bypassing nlpaug bug)")
for _, row in tqdm(df.iterrows(), total=len(df)):
    text = row['text']
    label = row['label']

    # Always preserve the original text row
    augmented_records.append({"text": text, "label": label})

    words = text.split()
    # Skip sentences that are too short to mask meaningfully
    if len(words) < 3:
        continue

    # Generate 30 variations per text sentence
    for _ in range(30):
        # Pick a random word index to replace (avoiding simple punctuation if possible)
        mask_idx = random.randint(0, len(words) - 1)
        original_word = words[mask_idx]

        # Insert the special [MASK] token into the text
        masked_words = words.copy()
        masked_words[mask_idx] = "[MASK]"
        masked_sentence = " ".join(masked_words)

        try:
            # Request top 5 alternative context words from the model
            predictions = mask_filler(masked_sentence, top_k=5)

            # Extract and save the newly generated sentences
            for pred in predictions:
                variant_text = pred['sequence']
                # Clean up any residual double spacing
                variant_text = " ".join(variant_text.split())
                augmented_records.append({"text": variant_text, "label": label})
        except Exception:
            pass

# 4. Clean up the final dataset structure
df_augmented = pd.DataFrame(augmented_records)
df_augmented.drop_duplicates(subset=['text'], inplace=True)

# 5. Save the output
df_augmented.to_csv("final_augmented_dataset.csv", index=False)

print("\n🚀 Expansion complete!")
print(f"New dataset size: {len(df_augmented)} rows across all 9 classes.")
print("Saved to: 'final_augmented_dataset.csv'")


Original dataset size: 92 rows.
Loading text augmentation model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Generating synthetic variants... (bypassing nlpaug bug)


100%|██████████| 92/92 [00:17<00:00,  5.36it/s]



🚀 Expansion complete!
New dataset size: 3196 rows across all 9 classes.
Saved to: 'final_augmented_dataset.csv'
